In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sys

sys.path.append("../")

!pip install pandas fiona shapely pyproj rtree 
!pip install contextily
!pip install geopandas
!pip install topojson

import geopandas as gpd
import contextily as ctx
import matplotlib.pyplot as plt

!pip install py7zr geopandas openpyxl tqdm s3fs ipywidgets --quiet
!pip install PyYAML --quiet
!pip install cartiflette --quiet
!pip install statsmodels


import statsmodels.api as sm
import statsmodels.formula.api as smf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 78.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [fiona]32m3/4 [fiona]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 39.1/39.1 MB 72.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [contextily]7 [rasterio]

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip

[notice] A new re

In [47]:
%pip install s3fs
import s3fs

fs=s3fs.S3FileSystem(client_kwargs={"endpoint_url":"https://minio.lab.sspcloud.fr"})
MY_BUCKET = "mgibault"
print(fs.ls(MY_BUCKET))


[notice] A new release of pip is available: 26.2.2 -> 26.3
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.
['mgibault/DS_RP_EMPLOI_LR_COMP_2023_data.csv']


In [22]:
with fs.open(f"{MY_BUCKET}/DS_RP_EMPLOI_LR_COMP_2023_data.csv","rb")as f:
    emploi= pd.read_csv(f,sep=";",dtype={"GEO": str,"EMPSTA_ENQ":str})
    emploi = emploi [['TIME_PERIOD','EMPSTA_ENQ','GEO','OBS_VALUE','PCS']]

emploi_dep = emploi.loc[emploi["PCS"] == "_T"] # on choisit de s'intéresser aux résultats agrégés par PCS.
# Si on veut une analyse plus fine, on pourra dans un second temps regarder par PCS...
emploi_dep = emploi_dep.reset_index(drop=True)
emploi_dep = emploi_dep.rename(columns={
    "TIME_PERIOD": "ANNEE",
    "EMPSTA_ENQ":"STATUT",
    "GEO":"DEP",
    "OBS_VALUE": "POPULATION"
})

emploi_dep= emploi_dep.drop(columns=["PCS"])

emploi_dep.head(20)

,ANNEE,STATUT,DEP,POPULATION
0,2012,1,001,5.684150e+06
1,2012,1T2,001,6.377119e+06
2,2012,2,001,6.929694e+05
3,2017,1,001,5.696479e+06
4,2017,1T2,001,6.407343e+06
5,2017,2,001,7.108635e+05
6,2023,1,001,5.997447e+06
7,2023,1T2,001,6.598552e+06
8,2023,2,001,6.011057e+05
9,2012,1,002,9.094996e+05


In [42]:
emploi_pivot = emploi_dep[emploi_dep['DEP'].str.len() == 3]
emploi_pivot= emploi_pivot.pivot_table(
    index=['ANNEE', 'DEP'],
    columns='STATUT',
    values='POPULATION').reset_index()

emploi_pivot.head(20)


STATUT,ANNEE,DEP,1,1T2,2
0,2012,001,5.684150e+06,6.377119e+06,692969.389410
1,2012,002,9.094996e+05,1.021020e+06,111520.659150
2,2012,003,6.850123e+05,7.922593e+05,107246.977690
3,2012,004,5.828589e+05,6.732480e+05,90389.106180
4,2012,005,5.833271e+05,6.553992e+05,72072.172470
5,2012,006,5.238807e+05,5.903065e+05,66425.825870
6,2012,008,4.030589e+05,4.481345e+05,45075.620400
7,2012,010,3.502405e+05,3.941662e+05,43925.765410
8,2012,011,5.005247e+04,5.536768e+04,5315.215620
9,2012,012,2.117370e+05,2.419748e+05,30237.769915


In [43]:
emploi_pivot["Taux_activite"]=emploi_pivot["1"]/emploi_pivot['1T2']
emploi_pivot=emploi_pivot.drop(columns=["1","1T2","2"])
emploi_pivot.head()

STATUT,ANNEE,DEP,Taux_activite
0,2012,001,0.891335
1,2012,002,0.890775
2,2012,003,0.864631
3,2012,004,0.865742
4,2012,005,0.890033


Préparation des données du bac à partir de ce site: 

In [66]:
with fs.open('mgibault/fr-en-indicateurs-de-resultat-des-lycees-denseignement-general-et-technologique (1).csv',"rb")as bac_f:
    bac= pd.read_csv(bac_f,sep=";")

FileNotFoundError: mgibault/fr-en-indicateurs-de-resultat-des-lycees-denseignement-general-et-technologique (1).csv

'mgibault/fr-en-indicateurs-de-resultat-des-lycees-denseignement-general-et-technologique (1).csv'
False
